# 2주차 — 신정호 강사님 풀이 (Blind)
- 문제지만 보고 작성. 정답지 미참조.
- 10개 문제 모두 TODO 구현.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import time
from typing import List, Dict, Any, Tuple, Callable

## Q1. Tokenizer: EOS/BOS + Padding

In [ ]:
def preprocess_sentences(sentences: List[List[int]], max_l: int) -> Tuple[torch.Tensor, torch.Tensor]:
    batch_ids = []
    batch_masks = []
    for s in sentences:
        wrapped = [1] + list(s) + [2]
        # 길이 부족분을 PAD(0)로 채움 (초과 시 잘라냄)
        if len(wrapped) < max_l:
            padded = wrapped + [0] * (max_l - len(wrapped))
            mask = [1] * len(wrapped) + [0] * (max_l - len(wrapped))
        else:
            padded = wrapped[:max_l]
            mask = [1] * max_l
        batch_ids.append(padded)
        batch_masks.append(mask)
    return torch.tensor(batch_ids, dtype=torch.long), torch.tensor(batch_masks, dtype=torch.long)

## Q2. Multi-Head Split

In [ ]:
def split_heads(tensor: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq, hidden = tensor.size()
    head_dim = hidden // num_heads
    # (B, S, H) -> (B, S, n_heads, head_dim) -> (B, n_heads, S, head_dim)
    return tensor.view(batch, seq, num_heads, head_dim).transpose(1, 2)

## Q3. KV Cache 누적

In [ ]:
def append_to_kv_cache(prev_k: torch.Tensor, new_k: torch.Tensor) -> torch.Tensor:
    return torch.cat([prev_k, new_k], dim=-2)

## Q4. Temperature Softmax

In [ ]:
def get_probs_with_temp(logits: torch.Tensor, temp: float) -> torch.Tensor:
    return F.softmax(logits / temp, dim=-1)

## Q5. Tool Schema

In [ ]:
currency_tool_schema = {
    "type": "object",
    "properties": {
        "amount":   {"type": "number", "description": "금액"},
        "from_cur": {"type": "string", "description": "원화폐 코드 (예: KRW)"},
        "to_cur":   {"type": "string", "description": "대상 화폐 코드", "default": "USD"},
    },
    "required": ["amount", "from_cur"],
}

## Q6. Model Call Parser

In [ ]:
def parse_model_call(call_str: str) -> Dict[str, Any]:
    m = re.match(r"\s*CALL:\s*([A-Za-z_][A-Za-z0-9_]*)\((.*)\)\s*$", call_str)
    if not m:
        return {}
    func_name = m.group(1)
    args_str = m.group(2).strip()
    args: Dict[str, Any] = {}
    if args_str:
        for pair in re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*([^,]+)", args_str):
            k, v = pair
            args[k] = v.strip().strip('\'"')
    return {"func_name": func_name, "args": args}

## Q7. MHA KV Cache Update

In [ ]:
def update_mha_kv_cache(prev_k: torch.Tensor, new_k_proj: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq_new, hidden = new_k_proj.size()
    head_dim = hidden // num_heads
    new_k = new_k_proj.view(batch, seq_new, num_heads, head_dim).transpose(1, 2)
    return torch.cat([prev_k, new_k], dim=-2)

## Q8. Multi-Step Tool Chaining

In [ ]:
def multi_step_tool_agent(item: str, tools: Dict[str, Callable]) -> str:
    pid = tools["find_product_id"](item)
    if pid is None:
        return "ERROR: PRODUCT_NOT_FOUND"
    rate = tools["get_discount_rate"](pid)
    if rate > 0.5:
        return "ERROR: INVALID_DISCOUNT"
    price = tools["calculate_price"](pid, rate)
    return f"Final Price for {item} is {price}"

## Q9. Sliding Window Batch KV

In [ ]:
def sliding_window_batch_kv(cache: torch.Tensor, new_val: torch.Tensor, window_size: int) -> torch.Tensor:
    # (B, H, S, D)의 S 차원으로 append → window 초과 시 앞쪽 잘라내기
    appended = torch.cat([cache, new_val], dim=-2)
    if appended.size(-2) > window_size:
        appended = appended[..., -window_size:, :]
    return appended

## Q10. LLM API Safety & Retry

In [ ]:
def call_llm_api_safely(prompt: str, max_tokens: int, api_fn: Callable) -> Dict[str, Any]:
    if len(prompt.split()) > max_tokens:
        return "ERROR: CONTEXT_EXCEEDED"

    for attempt in range(1, 4):
        status, body = api_fn(prompt)
        if status == 200:
            return {"status": "success", "response": body, "attempts": attempt}
        if status == 429:
            if attempt < 3:
                time.sleep(1)
                continue
            return {"status": "fail", "reason": "RATE_LIMIT_EXCEEDED"}
        # 기타 에러 (500 등)
        return {"status": "fail", "reason": "SERVER_ERROR"}
    return {"status": "fail", "reason": "RATE_LIMIT_EXCEEDED"}

## 검증 섹션

In [ ]:
print('--- 검증 결과 ---')

q1_res, _ = preprocess_sentences([[10, 20]], 5)
print('Q1:', q1_res.tolist() == [[1, 10, 20, 2, 0]])

p_k = torch.zeros(1, 2, 4, 8)
n_k_p = torch.ones(1, 1, 16)
res_7 = update_mha_kv_cache(p_k, n_k_p, 2)
print('Q7 (Shape):', res_7.shape == (1, 2, 5, 8))

m_tools = {
    'find_product_id': lambda n: 'p1' if n == 'Phone' else None,
    'get_discount_rate': lambda pid: 0.2,
    'calculate_price': lambda pid, r: int(500 * (1 - r)),
}
print('Q8 (Success):', multi_step_tool_agent('Phone', m_tools) == 'Final Price for Phone is 400')
print('Q8 (Fail):', multi_step_tool_agent('Desk', m_tools) == 'ERROR: PRODUCT_NOT_FOUND')

def mock_api(p): return (429, 'Limit')
res_10 = call_llm_api_safely('Check this prompt', 10, mock_api)
print('Q10 (Retry):', res_10.get('reason') == 'RATE_LIMIT_EXCEEDED')